# CyberGate on Qwen3.8-27B

Fit both gate factors on a model that has never been gated, then serve it and check the gate is
actually live.

**Hardware.** Qwen3.8-27B at bf16 is ~55 GB of weights. That needs an **A100 80 GB or B100** --
a 40 GB A100 or a 24 GB L4 will not hold it. Only 16 of its 64 layers are full-attention, so the
KV cache is cheap and 32k context costs ~2 GB.

**Runtime -> Change runtime type -> A100 / B100, High-RAM.**

### What this notebook does

| step | what | needs GPU |
|---|---|---|
| 1-3 | install, download weights, read the architecture | no |
| 4 | **measure the chat-template token IDs** | no |
| 5 | upload the fit corpora | no |
| 6 | declare the `qwen38` family | no |
| 7 | **factor 1**: category direction, in the serving render (+ layer sweep) | yes |
| 8 | **factor 2**: refusal direction and its sigma calibration | yes |
| 9 | assemble the kit | no |
| 10-11 | serve with the plugin, verify the gate is live | yes |

### Three things that will bite you if skipped

**The corpora are not in the repo** (step 5). The 240 fit prompts and the harmful/harmless pair
have to be uploaded.

**Steps 7 and 8 each load the model.** One at a time. Step 8 refuses to start unless the card is
free, because two 55 GB copies on an 80 GB card offload to CPU rather than fail, and the capture
then runs ~50x slower. **A kernel restart between them is normal.** Everything each step needs is
written to `/content/qwen38_state.json` and `/content/acts_qwen38*.npz` first, so a restart costs
nothing -- re-run cells 4, 7, 9, 13 and continue.

**Both factors are fitted in the serving render**, which is the render the server scores in. The
`fit` render (`apply_chat_template`) is what reproduces the shipped *gemma4* operator and is the
wrong choice for a new model: it fits a direction in a geometry the gate never sees.


## 1. Check the GPU

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)

import torch
free, total = torch.cuda.mem_get_info()
gb = total / 1024**3
print(f"visible GPU memory: {gb:.1f} GiB")
NEED_BF16 = 60
if gb < NEED_BF16:
    print(f"\n*** {gb:.0f} GiB is not enough for Qwen3.8-27B at bf16 (~55 GB of weights).")
    print("    Switch to an A100 80GB or B100. Fitting on a 4-bit load and serving at bf16 is NOT")
    print("    a workaround: a direction fitted on different weights scores near zero on the served")
    print("    ones and the gate just stops firing, with no error. That is the exact failure the")
    print("    kit canaries exist to catch.")
else:
    print(f"OK: {gb:.0f} GiB is enough for bf16.")

## 2. Install, and clone the repo

Set `REPO_URL` to your GitHub remote.

In [ ]:
REPO_URL = "https://github.com/kncshw/cybergate.git"   # public; no token needed
BRANCH   = "main"                                          # <-- EDIT if needed

import os, subprocess, sys, pathlib, time
import importlib.metadata as md

def _ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

if not pathlib.Path("/content/cybergate").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL,
                    "/content/cybergate"], check=True)
os.chdir("/content/cybergate")
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

torch_before = _ver("torch")
print(f"torch before: {torch_before}")
print("Installing vllm==0.32.0. This pulls torch and the nvidia-* CUDA wheels -- several GB.")
print("EXPECT 5-15 MINUTES WITH NO OUTPUT. It is not hung; -q suppresses pip's progress.",
      flush=True)

# vLLM pins torch; let it. The plugin itself declares NO runtime dependencies.
t0 = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.32.0"], check=True)
print(f"vllm installed in {time.time() - t0:.0f}s", flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
# Do NOT add --no-build-isolation: a venv on Python >=3.12 has no setuptools and it fails with
# ModuleNotFoundError: No module named 'setuptools'.
import cybergate
print("cybergate imported from", cybergate.__file__)

# vLLM pins a torch version. If pip replaced the one Colab preinstalled, the OLD module is still
# loaded in this kernel and every later cell that touches torch fails in ways that look like a
# CyberGate bug rather than a stale import.
torch_after = _ver("torch")
if torch_before is not None and torch_after != torch_before:
    raise RuntimeError(
        "\n\n=== RESTART REQUIRED -- THIS IS EXPECTED, NOT A FAILURE ===\n"
        f"pip replaced torch {torch_before} -> {torch_after} underneath a running kernel.\n"
        "Runtime -> Restart session, then re-run THIS cell and continue. The clone is skipped\n"
        "and pip is a no-op, so the re-run takes seconds.\n")
print(f"torch {torch_after} -- unchanged, no restart needed")


In [ ]:
# The offline suite should pass before anything expensive. No GPU, no network, no model.
!python -m pytest -q -p no:randomly -rN --tb=short 2>&1 | tail -5

## 3. Download the weights and read the architecture

Qwen3.8-27B is a **composite** config: the decoder shape is under `text_config`, beside a
`vision_config`. Reading `config.hidden_size` at the top level raises or, worse, picks up the
vision tower's width.

In [ ]:
MODEL = "Qwen/Qwen3.8-27B"      # or "Qwen/Qwen3.8-27B-FP8" for ~28 GB instead of ~55 GB

from huggingface_hub import snapshot_download
LOCAL = snapshot_download(MODEL, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model"])
print("weights at", LOCAL)

from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(LOCAL)
txt = getattr(cfg, "text_config", cfg)
HIDDEN = int(txt.hidden_size)
NLAYERS = int(txt.num_hidden_layers)
print(f"architectures   {getattr(cfg, 'architectures', None)}")
print(f"hidden_size     {HIDDEN}")
print(f"num_layers      {NLAYERS}")

lt = list(getattr(txt, "layer_types", []) or [])
if lt:
    full = [i for i, t in enumerate(lt) if t == "full_attention"]
    print(f"layer_types     {len(lt)} entries, {len(full)} full_attention at {full[:8]}{' ...' if len(full) > 8 else ''}")
    print("\nPrefer a full_attention layer for the gate. The other 48 are Gated DeltaNet (linear")
    print("attention); whether their hidden states carry the same linearly-decodable category")
    print("signal is UNKNOWN -- nobody has measured it. The sweep in step 7 is how you find out.")
    FULL_ATTN = full
else:
    FULL_ATTN = list(range(NLAYERS))

## 4. Measure the chat-template token IDs

**Do not skip and do not guess these.** They are checkpoint-specific. The canonical package
hardcoded Qwen2.5's `(151644, 872, 198)`; on the Qwen3.6 checkpoint `151644` decodes to ordinary
text, so the exact-marker branch never matched and span location silently fell back to fixed
offsets on every single request. The fix was to measure them -- Qwen3.6 turned out to be
`(248045, 846, 198)`.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(LOCAL)
enc = lambda s: tok(s, add_special_tokens=False)["input_ids"]

USER_TURN_MARKER  = tuple(enc("<|im_start|>user\n"))
TURN_END          = enc("<|im_end|>")
FRESH_PROMPT      = tuple(enc("<|im_start|>system"))
print(f"vocab_size         {len(tok)}")
print(f"user_turn_marker   {USER_TURN_MARKER}   <- <|im_start|>user\\n")
print(f"turn_end_id        {TURN_END}           <- <|im_end|>")
print(f"fresh_prompt_marker{FRESH_PROMPT}       <- <|im_start|>system")

assert len(TURN_END) == 1, f"<|im_end|> is not one token ({TURN_END}); this is not a ChatML template"
TURN_END_ID = TURN_END[0]

# Round-trip, so a silently-wrong id cannot pass.
assert tok.decode(list(USER_TURN_MARKER)).strip().endswith("user"), tok.decode(list(USER_TURN_MARKER))
print("\nrendered single turn:\n")
print(repr(tok.apply_chat_template([{"role": "system", "content": "You are a helpful assistant."},
                                    {"role": "user", "content": "PROBE"}],
                                   add_generation_prompt=True, tokenize=False)))

## 5. Upload the fit corpora

Four files. Rows may be `{"text": ...}` or `{"prompt": ...}`; bare strings work where no `cat` is
needed.

| file | rows | needs `cat` |
|---|---|---|
| `pos_train_cyber_100.json` | 100 target-category prompts | no |
| `neg_ref_14cats_140.json` | 140 non-target, **14 subcategories x 10** | **yes** |
| `harmful.json` | harmful prompts, for the refusal direction | no |
| `harmless.json` | harmless prompts; also the sigma calibration set | no |

The `cat` on the negatives is load-bearing: `M1_subcat` centres each subcategory on its own mean,
and that is the entire difference from the earlier `B0` arm.

In [ ]:
from google.colab import files
import json, pathlib, collections
pathlib.Path("/content/corpora").mkdir(exist_ok=True)
up = files.upload()
for name, blob in up.items():
    pathlib.Path("/content/corpora", name).write_bytes(blob)

def check(path, need_cat):
    rows = json.loads(pathlib.Path(path).read_text())
    rows = rows if isinstance(rows, list) else (rows.get("rows") or rows.get("prompts"))
    texts = [r if isinstance(r, str) else (r.get("text") or r.get("prompt")) for r in rows]
    assert all(texts), f"{path}: some rows have no text/prompt"
    msg = f"  {pathlib.Path(path).name:<32} {len(rows):>5} rows"
    if need_cat:
        cats = collections.Counter(r.get("cat") or r.get("category") for r in rows)
        assert None not in cats, f"{path}: rows without `cat`; M1_subcat needs it"
        msg += f"   {len(cats)} subcategories, {min(cats.values())}-{max(cats.values())} each"
    print(msg)
    return len(rows)

POS  = "/content/corpora/pos_train_cyber_100.json"
NEG  = "/content/corpora/neg_ref_14cats_140.json"
HARM = "/content/corpora/harmful.json"
HARMLESS = "/content/corpora/harmless.json"
for p, c in ((POS, False), (NEG, True), (HARM, False), (HARMLESS, False)):
    check(p, c)

# Every later step reloads what it needs from here rather than from a live variable. A kernel
# restart between the two 55 GB captures is normal -- and expected, see step 8 -- and without this
# file step 9 dies on `NameError: OP_NAME` with both captures already paid for.
STATE = "/content/qwen38_state.json"
pathlib.Path(STATE).write_text(json.dumps(
    {"pos": POS, "neg": NEG, "harm": HARM, "harmless": HARMLESS}, indent=1))
print(f"\nwrote {STATE} -- /content survives a kernel restart, only a runtime-type change wipes it")


## 6. Declare the `qwen38` family

This is the whole cost of adding a model: one class, no edits anywhere under `src/`. It subclasses
`Qwen36Family` to inherit the ChatML scaffolding and policy thresholds, and overrides only what is
genuinely different -- the measured token ids and the checkpoint shape.

`expected_gate`, `ref_*` and the operator filenames are filled in after the fits, in step 9.

In [ ]:
import os, pathlib, textwrap

# Every path below is relative, and mkdir/write_text succeed in ANY directory -- so a wrong cwd
# writes the generated family outside the repo, where nothing imports it, and only the later read
# of registry.py notices. A kernel restart clears step 2's chdir, which is exactly how that happens.
# Locate the repo from the installed package instead of trusting cwd.
import cybergate
REPO = pathlib.Path(cybergate.__file__).resolve().parents[2]
assert (REPO / "src/cybergate/families/registry.py").is_file(), (
    f"{REPO} does not look like the cybergate tree; re-run step 2")
os.chdir(REPO)
print(f"repo root {REPO}")

pathlib.Path("src/cybergate/families/qwen38").mkdir(parents=True, exist_ok=True)

GATE_LAYER_GUESS = min(FULL_ATTN, key=lambda i: abs(i - int(0.52 * NLAYERS)))
REF_LAYER_GUESS  = min(FULL_ATTN, key=lambda i: abs(i - int(0.70 * NLAYERS)))
print(f"gate layer candidate {GATE_LAYER_GUESS}  (~52% depth, qwen36 gates at L21 of 40)")
print(f"refusal layer candidate {REF_LAYER_GUESS}  (~70% depth, qwen36 monitors at L28 of 40)")

Q = chr(34) * 3
Q1 = chr(34)
# models--Qwen--Qwen3.8-27B style cache directory, derived from MODEL rather than hardcoded.
HF_GLOB = "/root/.cache/huggingface/hub/models--" + MODEL.replace("/", "--") + "/snapshots/*"
family_py = "\n".join([
    Q + "Qwen3.8-27B. Declared from measurements taken in notebooks/qwen38_colab.ipynb." + Q,
    "from __future__ import annotations",
    "",
    "from ..qwen36.family import Qwen36Family",
    "",
    "",
    "class Qwen38Family(Qwen36Family):",
    "    " + Q + f"Qwen3.8-27B: 27B dense, {NLAYERS} layers, hybrid full/linear attention.",
    "",
    "    Inherits Qwen3.6's ChatML scaffolding and policy thresholds. Overridden below is only",
    "    what differs: the token ids (measured on THIS checkpoint -- they are not portable",
    "    between Qwen releases) and the checkpoint shape.",
    "",
    "    NOT inherited-and-verified: the policy thresholds (revoke_ceil, hysteresis_thresh,",
    "    in_context_pivot_floor, s_cat_last_pivot_floor) are Qwen3.6's, calibrated at its",
    "    L15/L28. They have NOT been recalibrated here. Treat the mid/late veto and rescue",
    "    paths as unvalidated on this model until they are.",
    "    " + Q,
    "",
    '    name = "qwen38"',
    "    # Both matchers must be MORE SPECIFIC than qwen36's bare ('qwen',) and this family must be",
    "    # registered BEFORE qwen36, because both lookups are first-match-wins. Appending it after",
    "    # qwen36 leaves it unmatchable: 'Qwen/Qwen3.8-27B' contains 'qwen' and resolves to qwen36.",
    '    model_id_substrings = ("qwen3.8", "qwen3_8")',
    '    model_key_prefixes = ("qwen38",)',
    "    verified_live = False",
    "",
    "    # Measured from the tokenizer, not inherited. Qwen3.6 is (248045, 846, 198)/248046.",
    f"    user_turn_marker = {USER_TURN_MARKER!r}",
    f"    turn_end_id = {TURN_END_ID!r}",
    f"    fresh_prompt_marker = {FRESH_PROMPT!r}",
    '    span_label = "exact_token_markers(qwen38)"',
    "",
    f"    hidden_size = {HIDDEN}",
    f"    num_layers = {NLAYERS}",
    '    kit_name = "qwen38"',
    "",
    "    # Filled in by step 9 once the fits have run.",
    "    expected_gate = None",
    "    default_op_names = ()",
    "    default_ref_dir_name = None",
    "    default_bundle_name = None",
    "",
    "    # Declared, not inherited. `test_every_family_declares_every_serving_attribute_itself`",
    "    # requires all 30 on the class: every base default is a legal value, so an inherited one",
    "    # is read, used and reported as success. Values below are Qwen3.6's ChatML scaffolding,",
    "    # which cell 4 verified is this model's template too -- EXCEPT where noted.",
    "    turn_open = " + Q1 + "<|im_start|>" + Q1,
    "    turn_close = " + Q1 + "<|im_end|>" + Q1,
    '    assistant_role = "assistant"',
    "    supports_bos = False",
    '    bos_text = ""',
    '    think_system_prefix = ""',
    '    thought_open = "<think>\\n"',
    '    thought_close = "\\n</think>\\n\\n"',
    '    empty_thought_block = "<think>\\n\\n</think>\\n\\n"',
    '    thought_close_tag = "</think>"',
    '    stop_tokens = ("<|im_end|>",)',
    "",
    "    # UNVERIFIED on this checkpoint: Qwen3.6's fixed-offset fallback. Only consulted when the",
    "    # exact markers above are absent, which should not happen -- if `span_method` in a decision",
    "    # record ever reads default_offsets(14:-5), the markers did not match and the score is",
    "    # pooled over the wrong tokens. Re-measure rather than trust these.",
    "    default_pre = 14",
    "    default_suf = 5",
    "    bos_token_id = None          # a Qwen prefill has no BOS",
    "",
    "    # UNVERIFIED on this checkpoint: Qwen3.6's policy thresholds, calibrated at its L15/L28.",
    "    # Recalibrate before trusting the mid/late veto or the rescue on this model.",
    "    revoke_ceil = -0.0250",
    "    hysteresis_thresh = -0.0250",
    "    in_context_pivot_floor = -0.0300",
    "    s_cat_last_pivot_floor = 0.0100",
    "",
    "    # Empty on purpose: no candidate-layer bundle has been measured for this model, and empty",
    "    # disables the middle/late consensus entirely. That is the safe default -- inheriting",
    "    # Qwen3.6's layer list would attach probes at layers chosen for a different architecture.",
    "    bundle_probe_layers = ()",
    "    mid_late_probes = ()",
    '    precomputed_tag = "qwen38"',
    "",
    "    # Colab runs vllm in-process, so the worker tokenizer is in the ordinary HF cache.",
    f"    tokenizer_globs = ({HF_GLOB!r},)",
    "    tokenizer_dirs = ()",
    "",
])
pathlib.Path("src/cybergate/families/qwen38/family.py").write_text(family_py)
pathlib.Path("src/cybergate/families/qwen38/__init__.py").write_text(
    "from .family import Qwen38Family\n\n__all__ = [\"Qwen38Family\"]\n")

# Register it. These two literals are matched EXACTLY against registry.py, so each replace is
# asserted: a silent no-op here leaves qwen38 unregistered and the next line fails with an
# unhelpful KeyError instead of pointing at the real cause.
reg = pathlib.Path("src/cybergate/families/registry.py")
src = reg.read_text()
if "Qwen38Family" not in src:
    IMPORT_ANCHOR = "from .qwen36.family import Qwen36Family"
    LIST_ANCHOR = "ALL_FAMILIES: tuple[ModelFamily, ...] = (GEMMA4, QWEN36)"
    for anchor in (IMPORT_ANCHOR, LIST_ANCHOR):
        assert anchor in src, (
            "registry.py does not contain %r -- the file changed since this notebook was written; "
            "open it and add qwen38 by hand" % anchor)
    src = src.replace(IMPORT_ANCHOR, IMPORT_ANCHOR + "\nfrom .qwen38.family import Qwen38Family")
    # QWEN38 goes BEFORE QWEN36. Both lookups iterate ALL_FAMILIES and take the first match, and
    # qwen36 claims the bare substring "qwen", so appending qwen38 after it makes qwen38 dead:
    # family_for_model_id("Qwen/Qwen3.8-27B") would return qwen36 and render with its markers.
    src = src.replace(LIST_ANCHOR,
                      "QWEN38 = Qwen38Family()\n"
                      "ALL_FAMILIES: tuple[ModelFamily, ...] = (GEMMA4, QWEN38, QWEN36)")
    reg.write_text(src)
    assert "Qwen38Family" in reg.read_text(), "registry.py was not updated"

import importlib, cybergate.families as F
importlib.reload(F)
from cybergate.families import family_by_name
fam = family_by_name("qwen38")
print(f"\nregistered: {fam.name}  hidden={fam.hidden_size} layers={fam.num_layers} kit={fam.kit_name}")
from cybergate.families import family_for_model_id, family_for_model_key
assert family_for_model_id(MODEL).name == "qwen38", (
    f"{MODEL} resolves to {family_for_model_id(MODEL).name}, not qwen38 -- qwen38 must be "
    "registered BEFORE qwen36 in ALL_FAMILIES, because both lookups take the first match")
assert family_for_model_key("qwen36").name == "qwen36", (
    "reordering broke qwen36: its operators now resolve to qwen38")
print(f"resolves:   {MODEL} -> qwen38, and qwen36 operators still -> qwen36")
print("\nNOTE: re-running the step-2 pytest now reports 4 failures, and that is correct:")
print("  3x test_every_family_declares_every_serving_attribute_itself[ref_*-qwen38]")
print("     -> clear once step 9 appends ref_layer/ref_mu_hm/ref_sd_hm from the refusal fit.")
print("  1x test_every_remaining_family_is_live_verified")
print("     -> stays until you set verified_live = True, which step 11 earns. Do not set it")
print("        before the gate is confirmed live on this checkpoint.")

## 7. Factor 1 — the category direction

Two stages, because the gate layer for this model is genuinely unknown: capture several candidate
layers in **one** model load, then solve each from the cache. Re-fitting is seconds; recapturing is
a 55 GB model load.

### The capture uses the serving render, and that is not a detail

The gate scores `cos(h, c)` on activations produced by `family.build_raw_prompt` at serve time.
Capturing under `tokenizer.apply_chat_template` instead — the `fit` render that reproduces the
shipped gemma4 operator — fits the direction in a **different geometry**. It looks excellent
offline and then collapses when served: on gemma4 one prompt scores +0.047, +0.167 or +0.186
depending on the render. Fit where you score.

The cell asserts the span was located by the **measured token markers**. If it falls back to
fixed offsets it pools arbitrary tokens, raises nothing, and produces a plausible, useless
direction — so that is a hard failure here rather than a warning.

The fit is `M1_subcat_Tridge_lam16`. Three things about it are easy to get wrong and each produces
a plausible-looking direction that is quietly useless -- see `scripts/fit/README.md`:

* rows are **unit-normalised** before centring (the gate scores a cosine)
* "Tridge" ridges the **whole spectrum**; `k=15` only picks the ridge constant `alpha = lambda_16`
* `R3` is the 12th-largest reference score, **no margin**

Reproducing the shipped gemma4 operator from its own prompts with this code gives cosine
**0.99946**, so the implementation is pinned to something real.

The AUC the next cell prints is **in-sample**, on 240 rows in a 5120-dimensional space. A ridge fit
separates almost any labelling at those proportions. It ranks the candidate layers; it is not
evidence the direction generalises, and the first out-of-sample test in this notebook is step 11.


In [ ]:
CANDIDATE_LAYERS = sorted({min(FULL_ATTN, key=lambda i: abs(i - int(f * NLAYERS)))
                           for f in (0.40, 0.48, 0.52, 0.60, 0.70)})
print("sweeping gate layers:", CANDIDATE_LAYERS)

import sys, json, pathlib, numpy as np, torch
sys.path.insert(0, "scripts/fit")
from fit_category import fit_m1_subcat_tridge, threshold_r3, load_prompts
from transformers import AutoModelForCausalLM
from cybergate.families import family_by_name

STATE = "/content/qwen38_state.json"
st = json.loads(pathlib.Path(STATE).read_text())
POS, NEG = st["pos"], st["neg"]

# Pre-flight (CLAUDE.md s7): a second 55 GB copy on an 80 GB card does not fail, it offloads to
# CPU and keeps going at roughly 1/50th speed. That is what "this one looks way slower" was.
free, total = torch.cuda.mem_get_info()
print(f"GPU free {free/1024**3:.1f} / {total/1024**3:.1f} GiB")
assert free / 1024**3 > 60, (
    "under 60 GiB free -- something still holds the GPU. Runtime -> Restart session, then re-run "
    "cells 4, 7, 9, 13 and this one.")

FAM = family_by_name("qwen38")
pos, neg = load_prompts(POS), load_prompts(NEG)
model = AutoModelForCausalLM.from_pretrained(LOCAL, dtype=torch.bfloat16, device_map="auto",
                                             low_cpu_mem_usage=True).eval()
dev = next(model.parameters()).device
offloaded = [n for n, p in model.named_parameters() if p.is_meta]
assert not offloaded, (
    f"{len(offloaded)} parameters are on the meta device, so this card could not hold the model. "
    "The capture would still be numerically correct but ~50x slower; use an 80 GB card.")

# THE SERVING RENDER, and this is the whole point of the cell.
#
# The gate scores cos(h, c) on activations produced by `build_raw_prompt` at serve time. Capturing
# under `apply_chat_template` instead (the `fit` render) fits a direction in a different geometry:
# it looks excellent offline and collapses when served. On gemma4 one prompt scores +0.047, +0.167
# or +0.186 depending on the render. Fit where you score.
def render_serving(text):
    raw = FAM.build_raw_prompt([{"role": "user", "content": text}], mode="baseline")
    ids = tok(raw, return_tensors="pt", add_special_tokens=True).input_ids
    start, end, method = FAM.locate_latest_user_span(ids[0].tolist())
    if end <= start:
        raise RuntimeError(f"empty user span ({start},{end}) via {method!r}")
    return ids, (start, end), method

_ids, _span, _method = render_serving(pos[0]["prompt"])
print(f"span method  {_method}")
print(f"first prompt {_ids.shape[1]} tokens, user span {_span}")
assert _method.startswith("exact_token_markers"), (
    f"the span was located by {_method!r}, not by the measured token markers. Cell 9's marker ids "
    "are wrong for this checkpoint and the capture would pool arbitrary tokens -- the exact "
    "silent failure the kit canaries exist to catch. Fix step 4 before capturing.")

def capture(rows, label):
    '''(n, len(CANDIDATE_LAYERS), hidden) pooled over the user span, one model pass per prompt.'''
    out = np.zeros((len(rows), len(CANDIDATE_LAYERS), HIDDEN), dtype=np.float32)
    for i, r in enumerate(rows):
        ids, span, _ = render_serving(r["prompt"])
        with torch.no_grad():
            hs = model(ids.to(dev), output_hidden_states=True).hidden_states
        for j, L in enumerate(CANDIDATE_LAYERS):
            h = hs[L + 1][0].to(torch.float32).cpu().numpy()   # acts[L] = hidden_states[L+1]
            out[i, j] = h[span[0]:span[1]].mean(axis=0)
        if (i + 1) % 25 == 0:
            print(f"  {label} {i+1}/{len(rows)}", flush=True)
    return out

A_pos, A_neg = capture(pos, "pos"), capture(neg, "neg")
np.savez_compressed("/content/acts_qwen38.npz", pos=A_pos, neg=A_neg,
                    layers=np.array(CANDIDATE_LAYERS))
print("captured ->", A_pos.shape, A_neg.shape, "-> /content/acts_qwen38.npz")


In [ ]:
# Solve every candidate layer from the cache and pick one on evidence.
cats = [r["cat"] for r in neg]
rows = []
for j, L in enumerate(CANDIDATE_LAYERS):
    P, N = A_pos[:, j, :].astype(np.float64), A_neg[:, j, :].astype(np.float64)
    c, diag = fit_m1_subcat_tridge(P, N, cats, k=15)
    tau, ref_scores = threshold_r3(c, N)
    s_pos = (P @ c) / np.linalg.norm(P, axis=1)
    s_neg = np.array(ref_scores)
    # AUC by rank, positives vs reference negatives
    allv = np.concatenate([s_pos, s_neg]); order = allv.argsort().argsort() + 1
    r_pos = order[:len(s_pos)].sum()
    auc = (r_pos - len(s_pos) * (len(s_pos) + 1) / 2) / (len(s_pos) * len(s_neg))
    rows.append((auc, L, tau, int((s_pos >= tau).sum()), diag))
    print(f"  L{L:<3} AUC={auc:.4f}  tau={tau:+.8f}  fit positives firing {int((s_pos>=tau).sum())}/{len(s_pos)}"
          f"  alpha={diag['k_or_alpha']['alpha']:.3g}")

rows.sort(reverse=True)
BEST_AUC, GATE_LAYER, GATE_TAU, _, _ = rows[0]
print(f"\nchosen gate layer: L{GATE_LAYER}  (AUC {BEST_AUC:.4f}, tau {GATE_TAU:+.8f})")
print("AUC here is IN-SAMPLE on the fit prompts -- it ranks layers, it does not estimate recall.")

In [ ]:
# Write the chosen operator.
import pathlib, gc
pathlib.Path("kits/qwen38").mkdir(parents=True, exist_ok=True)
j = CANDIDATE_LAYERS.index(GATE_LAYER)
P, N = A_pos[:, j, :].astype(np.float64), A_neg[:, j, :].astype(np.float64)
c_unit, diag = fit_m1_subcat_tridge(P, N, cats, k=15)
tau, _ = threshold_r3(c_unit, N)
meta = dict(diag)
# `model_key` is how the WORKER hook picks the family, by prefix match. It must be this family's
# own prefix. "qwen" matches qwen36 first -- qwen36 claims the bare "qwen" -- and the 5120-dim
# vector then fails qwen36's 2048-dim declaration with
#   KitMismatch: operator vector is 5120-dimensional but family 'qwen36' was fitted on 2048
# at serve time, after the whole capture is paid for.
meta.update({"gate_layer": GATE_LAYER, "pool": "mean_user", "category": "cyber_offense",
             "model_key": "qwen38", "model_tag": "qwen38", "serve_thresh_rule": "R3",
             "m_budget": 11, "thresholds_all": {"R3": tau}, "fit_render": "serving",
             "fit_system_prompt": None, "fit_model": MODEL,
             "fitted_by": "notebooks/qwen38_colab.ipynb"})
assert meta["model_key"] in FAM.model_key_prefixes, (
    f"model_key {meta['model_key']!r} is not one of qwen38's {FAM.model_key_prefixes}")
OP_NAME = f"qwen38_L{GATE_LAYER}_M1_serving.json"
pathlib.Path("kits/qwen38", OP_NAME).write_text(json.dumps(
    {"c": [float(x) for x in c_unit], "thresh": tau, "meta": meta}, indent=1))
print("wrote kits/qwen38/" + OP_NAME)

st.update({"gate_layer": int(GATE_LAYER), "op_name": OP_NAME, "tau": float(tau),
           "candidate_layers": [int(L) for L in CANDIDATE_LAYERS]})
pathlib.Path(STATE).write_text(json.dumps(st, indent=1))
print(f"state -> {STATE}")

# Release the card before step 8, which loads the model again in a subprocess. A bare `del` is not
# enough: IPython holds the last results in `_`, `__`, `___` and `Out`, and any one of those keeps
# the 55 GB alive. Step 8 then offloads instead of failing, and crawls.
del model, A_pos, A_neg
for _n in ("_", "__", "___"):
    globals().pop(_n, None)
try:
    get_ipython().user_ns["Out"].clear()
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free {free/1024**3:.1f} / {total/1024**3:.1f} GiB after release")
if free / 1024**3 < 60:
    print("\n*** STILL HELD. Runtime -> Restart session, then re-run cells 4, 7, 9, 13, then step 8.")
    print("    Nothing is lost: the capture is in /content/acts_qwen38.npz, the operator is")
    print("    written, and /content/qwen38_state.json carries everything step 9 needs.")


## 8. Factor 2 — the refusal direction

Without this the gate runs on **one** factor. A family with no refusal direction falls back to
`mu=0.0, sd=1.0`, which leaves `s_ref_z` an uncalibrated raw cosine, and the policy's `1.50` sigma
floor is then effectively unreachable.

`mu_hm_cos` and `sd_hm_cos` *are* the sigma scale: `s_ref_z = (s_ref_cos - mu_hm_cos)/sd_hm_cos`.

**This step loads the model a second time, in a subprocess.** The first cell below refuses to
continue unless the card is actually free. Two 55 GB copies on an 80 GB card do not fail — they
offload to CPU and run at roughly 1/50th speed, which is what "this one looks way slower" was. If
it stops you, restart the runtime and re-run cells 4, 7, 9, 13; nothing is lost, because the
category capture, the operator and `/content/qwen38_state.json` are all on disk.

### Two numbers to read off the output

The direction is a difference of means **orthogonalized against the harmless centroid**, so
`<r_unit, gd>` should print as ~1e-16. If it does not, the fit is not doing what it claims.

`sep = ||harmful mean - harmless centroid|| / harmless spread` is how the original picked the
refusal layer: it swept every layer and took the peak. `REF_LAYER` below is a **70%-depth
guess**, not a measurement. gemma4 sits at `sep` 1.16 on the full corpus at its L21. If the
number here is much lower, re-run at a neighbouring full-attention layer and keep the better
one -- the capture is the slow part and `--acts-cache` makes the re-solve free.

`mu_hm_cos` should land near **0** for a full-corpus fit. gemma4's shipped `+0.137` is an
artifact of its 5-prompt direction being calibrated on a different corpus; see
`scripts/fit/README.md`. A value far from 0 here means the two corpora disagree.


In [ ]:
import json, pathlib, torch
STATE = "/content/qwen38_state.json"
st = json.loads(pathlib.Path(STATE).read_text())
HARM, HARMLESS = st["harm"], st["harmless"]
REF_LAYER = min(FULL_ATTN, key=lambda i: abs(i - int(0.70 * NLAYERS)))
REF_NAME = f"qwen38_L{REF_LAYER}_refusal_dir_serving.json"
print(f"fitting the refusal direction at L{REF_LAYER}, last-token pooling, SERVING render")

# fit_refusal.py loads its own copy of the model in a subprocess. If this kernel still holds one,
# that is two 55 GB models on an 80 GB card: no error, just CPU offload and ~7 min per 50 prompts.
free, total = torch.cuda.mem_get_info()
print(f"GPU free {free/1024**3:.1f} / {total/1024**3:.1f} GiB")
assert free / 1024**3 > 60, (
    "the GPU is still held by this kernel. Runtime -> Restart session, re-run cells 4, 7, 9, 13, "
    "then this cell. The category capture and the operator are already on disk.")

st["ref_layer"] = int(REF_LAYER); st["ref_name"] = REF_NAME
pathlib.Path(STATE).write_text(json.dumps(st, indent=1))


In [ ]:
!python scripts/fit/fit_refusal.py \
    --model {LOCAL} --family qwen38 --layer {REF_LAYER} --pool last --render serving \
    --harmful {HARM} --harmless {HARMLESS} \
    --out kits/qwen38/{REF_NAME} \
    --acts-cache /content/acts_qwen38_refusal.npz


## 9. Assemble the kit and finish the family declaration

In [ ]:
import json, pathlib, numpy as np

# Everything here is reloaded from disk, not from a live variable: a kernel restart between the
# two captures is normal and this cell used to die on `NameError: OP_NAME` with both captures
# already paid for.
STATE = "/content/qwen38_state.json"
st = json.loads(pathlib.Path(STATE).read_text())
GATE_LAYER, OP_NAME, tau = st["gate_layer"], st["op_name"], st["tau"]
REF_LAYER, ref_name = st["ref_layer"], st["ref_name"]
CANDIDATE_LAYERS = st["candidate_layers"]

import sys
sys.path.insert(0, "scripts/fit")
from fit_category import load_prompts
pos, neg = load_prompts(st["pos"]), load_prompts(st["neg"])

z = np.load("/content/acts_qwen38.npz")
j = list(z["layers"]).index(GATE_LAYER)
P = z["pos"][:, j, :].astype(np.float64)
N = z["neg"][:, j, :].astype(np.float64)
op = json.loads(pathlib.Path("kits/qwen38", OP_NAME).read_text())
c_unit = np.array(op["c"], dtype=np.float64)
ref = json.loads(pathlib.Path("kits/qwen38", ref_name).read_text())
rm = ref["meta"]
assert op["meta"]["fit_render"] == rm["fit_render"] == "serving", (
    "both factors must be fitted in the serving render for the canaries below to be meaningful")

manifest = {
    "kit": "qwen38", "model_ids": [MODEL],
    "files": {"operator": OP_NAME, "refusal_direction": ref_name, "canaries": "canaries.json"},
    "gate": {"layer": GATE_LAYER, "pool": "mean_user", "threshold": tau,
             "threshold_rule": "R3 (m_budget=11)", "fit_render": "serving"},
    "serve": {"reasoning_parser": None,
              "notes": "bf16; no fp8 and no TRITON_ATTN -- hybrid linear attention"},
    "fit": {"method": "M1_subcat_Tridge_lam16", "n_pos": len(pos), "n_neg": len(neg),
            "render": "serving", "fitted_by": "notebooks/qwen38_colab.ipynb"},
    "known_gaps": [
        "NOT measured on any evaluation corpus. The threshold is R3, which never sees one, so it "
        "is blind -- but recall and leak are simply unknown until a harvest is run.",
        "Every fit number is IN-SAMPLE: 240 rows solved in a 5120-dimensional space, where a "
        "ridge fit separates almost any labelling. The AUC ranks candidate layers; it does not "
        "estimate recall, and it is not evidence the direction generalises.",
        "The policy thresholds are inherited from Qwen3.6 and have NOT been recalibrated, so the "
        "mid/late veto and rescue paths are unvalidated on this model.",
        "Whether a hybrid linear-attention (Gated DeltaNet) stack is a sensible place to probe is "
        "unestablished. Only 16 of 64 layers are full attention; the gate layer is one of those.",
    ],
}
pathlib.Path("kits/qwen38/MANIFEST.json").write_text(json.dumps(manifest, indent=2))

# Canaries: highest-scoring fit positives, widest-margin quiet negatives. These scores are in the
# SERVING render -- the same render the server will use -- so step 11 replaying them is a real
# check. Under the old fit render they could not be, and the note here said so.
s_pos = (P @ c_unit) / np.linalg.norm(P, axis=1)
s_neg = (N @ c_unit) / np.linalg.norm(N, axis=1)
pos_idx = np.argsort(s_pos)[::-1][:5]
neg_idx = np.argsort(s_neg)[:4]
canaries = {
    "kit": "qwen38", "render": "cybergate_mode=baseline, no system prompt, max_tokens=1, temp=0",
    "note": "Scores measured in the SERVING render (family.build_raw_prompt + "
            "locate_latest_user_span), captured through transformers. Step 11 replays them "
            "through vLLM; agreement is the check that the gate is live on this checkpoint.",
    "canaries": (
        [{"role": "positive", "key": pos[i].get("key") or f"pos-{i}", "prompt": pos[i]["prompt"],
          "expect_fired": True, "measured_score": float(s_pos[i])} for i in pos_idx] +
        [{"role": "negative", "key": neg[i].get("key") or f"neg-{i}", "prompt": neg[i]["prompt"],
          "expect_fired": False, "measured_score": float(s_neg[i])} for i in neg_idx]),
}
pathlib.Path("kits/qwen38/canaries.json").write_text(json.dumps(canaries, indent=2))
print(f"kit written: {OP_NAME}, {ref_name}, MANIFEST.json, canaries.json")

# finish the family declaration
fp = pathlib.Path("src/cybergate/families/qwen38/family.py")
src = fp.read_text()
assert "expected_gate = None" in src and "ref_layer = " not in src, (
    "family.py no longer carries the placeholders, so this cell has already run. Re-run cell 13 "
    "to regenerate it before running this one, or the declaration gets two copies of ref_layer.")
src = src.replace("    expected_gate = None",
                  f'    expected_gate = ({GATE_LAYER}, "mean_user")')
src = src.replace("    default_op_names = ()", f'    default_op_names = ("{OP_NAME}",)')
src = src.replace("    default_ref_dir_name = None", f'    default_ref_dir_name = "{ref_name}"')
src += f'''
    # Refusal calibration, measured by scripts/fit/fit_refusal.py in this notebook.
    ref_layer = {REF_LAYER}
    ref_mu_hm = {rm["mu_hm_cos"]!r}
    ref_sd_hm = {rm["sd_hm_cos"]!r}
'''
fp.write_text(src)
print(open(fp).read())


In [ ]:
# The kit check must pass before serving: it is what catches a wrong-architecture operator.
import os, subprocess
os.environ.pop("CYBERGATE_SKIP_KIT_CHECK", None)
check_src = (
    "from cybergate.ops.loader import load_cybergate_op\n"
    "from cybergate.ops.kit import check_operator\n"
    "from cybergate.families import family_by_name\n"
    "import sys\n"
    "fam = family_by_name('qwen38')\n"
    "for p in sys.argv[1:]:\n"
    "    op = load_cybergate_op(p)\n"
    "    problems = check_operator(op, fam)\n"
    "    print('  %s: L%d %s dim=%d -> %s' % (p, op.gate_layer, op.pool, op.dim,\n"
    "          '; '.join(problems) if problems else 'clean'))\n"
)
r = subprocess.run(["python", "-c", check_src,
                    f"kits/qwen38/{OP_NAME}", f"kits/qwen38/{ref_name}"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
print("A refusal direction is exempt from the gate-layer check by design -- it is last-pooled at")
print("its own layer. It is still checked for width and layer count.")

## 10. Serve it with the plugin

The flags are load-bearing, and two of them differ from the Gemma recipe:

| flag | why |
|---|---|
| `--enforce-eager` | the hooks read hidden states; compiled graphs elide them |
| `--no-enable-prefix-caching` | **the subtle one.** A cached prefix skips the prefill forward pass, so the hook never runs and the decision is silently stale |
| *no* `--quantization` | bf16; the weights fit on an 80 GB card |
| *no* `--attention-backend` | TRITON_ATTN is wrong here. 48 of 64 layers are Gated DeltaNet; let vLLM choose |

`--enforce-eager` over 48 recurrent layers is the untested part of this whole notebook. If the
server never reaches `Application startup complete`, that is the first suspect.

### Three pre-flights, each one a failure this notebook actually hit

1. **`pkill -f vllm` and a free-memory check.** A leftover server owns port 8000 (`OSError:
   [Errno 98] Address already in use`) and owns the GPU, which makes this load offload instead
   of fail.
2. **The artifacts must exist.** Serving a kit whose files were never written gives
   `source=no_live_gpu_hook_attached` on every record and a server that looks perfectly healthy.
3. **The IPC directory is emptied.** A stale record *shadows the probe* — `_probe_observations`
   skips scoring entirely when a record already carries `s_ref_z` — so last run's numbers come
   back identical to six decimal places. That reads as stability and is result reuse.

`DIAGNOSE = True` additionally scores the shipped direction at **every** layer and dumps the
pooled vectors. Leave it on for this run; step 11 uses it to tell a layer-numbering mismatch
apart from a genuine activation difference.


In [ ]:
import os, subprocess, pathlib, time, json, gc, torch

# Set False once the gate is confirmed. It attaches a probe at EVERY layer, which costs prefill
# time, and dumps a pooled vector per layer per request (~650 KB each) -- cheap for a canary
# replay, not for a corpus harvest. What it buys: if the gate still does not fire, the record
# carries the shipped direction's score at all 64 layers, which says whether transformers and
# vLLM disagree about layer numbering or about the activations themselves.
DIAGNOSE = True

STATE = "/content/qwen38_state.json"
st = json.loads(pathlib.Path(STATE).read_text())
OP_NAME, ref_name, tau = st["op_name"], st["ref_name"], st["tau"]
IPC = "/dev/shm/cybergate_ipc_colab"
LOG = "/content/qwen38_serve.log"

# 1. A vLLM from an earlier attempt still owns port 8000 -> `OSError: [Errno 98] Address already
#    in use`, and still owns the GPU -> this load silently offloads to CPU.
subprocess.run(["pkill", "-f", "vllm"], check=False)
time.sleep(5)
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free {free/1024**3:.1f} / {total/1024**3:.1f} GiB")
assert free / 1024**3 > 60, (
    "the GPU is still held. Runtime -> Restart session, re-run cells 4, 7, 9, 13, then this cell.")

# 2. Serving an artifact that was never written gives `source=no_live_gpu_hook_attached` and a
#    server that looks healthy. Check before paying for the load.
for f in (OP_NAME, ref_name):
    p = pathlib.Path("kits/qwen38", f)
    assert p.is_file() and p.stat().st_size > 0, f"kits/qwen38/{f} is missing -- step 7/8 or 9 did not finish"

# 3. Stale IPC records SHADOW the probe. `_probe_observations` skips the probe entirely for a
#    record that already carries `s_ref_z`, so the previous run's scores reappear -- identical to
#    six decimal places, which reads as stability and is actually result reuse.
pathlib.Path(IPC).mkdir(parents=True, exist_ok=True)
stale = list(pathlib.Path(IPC).glob("*.json"))
for old in stale:
    old.unlink()
print(f"cleared {len(stale)} stale IPC record(s) from {IPC}")

env = {**os.environ,
       "CYBERGATE_OPS_DIR": "/content/cybergate/kits/qwen38",
       "CYBERGATE_OP_PATH": f"/content/cybergate/kits/qwen38/{OP_NAME}",
       "CYBERGATE_REF_DIR_PATH": f"/content/cybergate/kits/qwen38/{ref_name}",
       "CYBERGATE_MODE": "closed_loop",
       "CYBERGATE_IPC_DIR": IPC,
       "CYBERGATE_EVENTS_LOG_PATH": "/content/qwen38_events.log",
       "CYBERGATE_MAX_MODEL_LEN": "32768",
       "VLLM_USE_V2_MODEL_RUNNER": "0"}
if DIAGNOSE:
    env["CYBERGATE_CROSS_LAYER_DIAG"] = "1"
    env["CYBERGATE_DUMP_ACTS_DIR"] = "/content/acts_vllm"
    print("diagnostics ARMED: per-layer scores -> record['multi_layer_scores'],",
          "pooled vectors -> /content/acts_vllm")

cmd = ["vllm", "serve", LOCAL, "--host", "127.0.0.1", "--port", "8000",
       "--served-model-name", MODEL,
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.88",
       "--enforce-eager", "--no-enable-prefix-caching",
       "--middleware", "cybergate_plugin.middleware.CyberGateThinkMiddleware"]
print(" ".join(cmd))
log = open(LOG, "w")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
print("serving in background, pid", proc.pid)


In [ ]:
# Wait for startup, and watch for the two lines that say the gate attached.
import time, pathlib, re
for i in range(240):
    txt = pathlib.Path(LOG).read_text(errors="ignore")
    if "Application startup complete" in txt:
        print(f"READY after ~{i*5}s"); break
    if proc.poll() is not None:
        print("SERVER DIED. last 40 lines:\n"); print("\n".join(txt.splitlines()[-40:])); break
    time.sleep(5)

txt = pathlib.Path(LOG).read_text(errors="ignore")
for pat, good in (("CYBERGATE-KIT-MISMATCH", False), ("CYBERGATE-PLUGIN-WARN", False),
                  ("MONITOR", True), ("hook failed", False)):
    hits = [l for l in txt.splitlines() if pat in l]
    tag = "ok  " if bool(hits) == good else "LOOK"
    print(f"  [{tag}] {pat:<24} {len(hits)} line(s)")
    for l in hits[:2]:
        print("         ", l[:150])

## 11. Verify the gate is actually live

A server that answers proves nothing. The hooks run in the model-worker process and fail silently.
What proves the gate works is the decision record.

Capture and serve now use the **same render**, so the live score should track the captured one and
a disagreement is a real finding rather than a render artifact. That is what makes this replay a
check: previously the canaries were measured under `apply_chat_template` and could not be compared
with a served score at all.

Three things have to hold: a record **exists** for every prompt (the probe ran), `s_ref_z` is
**present** (the refusal monitor attached, so the gate has both factors), and the live scores
**track** the captured ones.


In [ ]:
import hashlib, json, pathlib, urllib.request, time

STATE = "/content/qwen38_state.json"
st = json.loads(pathlib.Path(STATE).read_text())
tau = st["tau"]
GATE_LAYER = st["gate_layer"]
IPC = "/dev/shm/cybergate_ipc_colab"   # not inherited: this cell must stand alone after a re-run

def ask(prompt, max_tokens=1, baseline=True):
    body = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "temperature": 0}
    if baseline:
        body["cybergate_mode"] = "baseline"
    req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=180).read())

def record_for(prompt):
    '''By sha of the decoded user span -- never the newest file, which races other traffic.'''
    sha = hashlib.sha256(prompt.encode()).hexdigest()
    for n in (f"sha_st_{sha}.json", f"sha_{sha}.json"):
        p = pathlib.Path(IPC, n)
        if p.is_file():
            return json.loads(p.read_text())
    return {}

cans = json.loads(pathlib.Path("kits/qwen38/canaries.json").read_text())["canaries"]
ok = 0
rows = []
for c in cans:
    ask(c["prompt"]); time.sleep(1)
    r = record_for(c["prompt"])
    if not r:
        print(f"  [FAIL] {c['key']:<24} no decision record -- the probe did not run"); continue
    got = bool(r.get("fired"))
    hit = got == c["expect_fired"]; ok += hit
    rows.append((c, r))
    print(f"  [{'ok  ' if hit else 'LOOK'}] {c['role']:<8} {c['key']:<22} "
          f"expect={c['expect_fired']!s:<5} got={got!s:<5} "
          f"live={r.get('score'):+.6f} capture={c['measured_score']:+.6f} "
          f"d={r.get('score', 0) - c['measured_score']:+.6f} s_ref_z={r.get('s_ref_z')}")
print(f"\n  {ok}/{len(cans)} canaries agreed.")

# Both sides are now the SERVING render, so the live score should track the captured one. A large
# `d` is a real finding, not a render artifact -- that is what changed in this notebook.
drift = [abs(r.get("score", 0) - c["measured_score"]) for c, r in rows]
if drift:
    print(f"  mean |live - capture| = {sum(drift)/len(drift):.6f}")
    if max(drift) > 0.05:
        print("""
  SCORES DID NOT TRANSFER. Capture and serve now use the same render and the same span, so this
  is the forward pass or the layer numbering. The record carries the answer if DIAGNOSE was on:
""")
        for c, r in rows[:1]:
            xl = {int(k[1:-len("_mean_user")]): v
                  for k, v in (r.get("multi_layer_scores") or {}).items()
                  if k.startswith("X") and k.endswith("_mean_user")}
            if not xl:
                print("    no X<layer>_mean_user keys -- set DIAGNOSE = True in step 10 and re-serve.")
            else:
                want = c["measured_score"]
                best = min(xl, key=lambda L: abs(xl[L] - want))
                print(f"    shipped direction scored at {len(xl)} layers; closest to the captured")
                print(f"    {want:+.6f} is L{best} at {xl[best]:+.6f} (gate is L{GATE_LAYER}).")
                if abs(xl[best] - want) < 0.02 and best != GATE_LAYER:
                    print(f"    -> LAYER NUMBERING MISMATCH. transformers L{GATE_LAYER} is vLLM L{best}.")
                    print(f"       Re-point the kit at L{best} and re-run from step 9.")
                else:
                    print("    -> no layer reproduces it, so numbering is not the cause. Re-fit from")
                    print("       the vectors in /content/acts_vllm, which are vLLM's own.")


In [ ]:
# Separation in the serving render -- the number that describes this operator live.
import numpy as np, time
sp, sn = [], []
for c in cans:
    ask(c["prompt"]); time.sleep(0.5)
    r = record_for(c["prompt"])
    if r.get("score") is None: continue
    (sp if c["expect_fired"] else sn).append(r["score"])
print(f"  positives (live): {np.round(sp, 6)}")
print(f"  negatives (live): {np.round(sn, 6)}")
if sp and sn:
    print(f"  gap: min(pos) {min(sp):+.6f}  vs  max(neg) {max(sn):+.6f}  "
          f"-> {'SEPARATED' if min(sp) > max(sn) else 'OVERLAPPING'}")
    print(f"  threshold {tau:+.8f} sits "
          f"{'inside' if max(sn) < tau < min(sp) else 'OUTSIDE'} that gap")
    if min(sp) > max(sn) and max(sn) < tau < min(sp):
        print("\n  The gate separates and the threshold is in the gap. Set")
        print("  `verified_live = True` on Qwen38Family and populate canaries.json from THESE")
        print("  live scores -- that is what earns the flag.")


## What is and is not established when this finishes

**Established:** both factors fit on this checkpoint **in the render the server uses**; the plugin
loads; the hooks attach; the probe runs on real activations; the refusal monitor is live so the
gate has two factors; and the canaries are comparable between capture and serve.

**Not established:**

* **recall and leak.** No evaluation corpus has been run. R3 never sees one, so the threshold is
  blind rather than in-sample -- but blind is not the same as measured.
* **generalisation.** Every fit number is in-sample: 240 rows in 5120 dimensions, where a ridge
  fit separates almost any labelling. The layer sweep ranks layers; it does not validate one.
  Step 11 is the first out-of-sample evidence in the whole notebook, and it is nine prompts.
* **the policy thresholds.** `revoke_ceil`, `hysteresis_thresh` and the pivot floors are inherited
  from Qwen3.6 at its L15/L28. The mid/late veto and rescue are unvalidated here. On gemma4 the
  rescue supplies 82 of 558 recorded fires -- 15% of delivered recall -- so this is not a detail.
* **whether a Gated DeltaNet layer is a sensible place to probe.** Only 16 of the 64 layers are
  full attention and the gate sits on one of them, but every layer's hidden state is downstream of
  48 recurrent ones.

To close the first gap, run the harvest from `scripts/serving/check_eval.sh` and an evaluation
corpus, and record the result in `kits/qwen38/MANIFEST.json`.

`verified_live = True` is earned by step 11 separating and the threshold landing inside the gap --
not by the server starting.
